In [ ]:
# TensorFlow
import tensorflow as tf

# Numerical operations
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# File handling
import os

# Image preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# EfficientNet model
from tensorflow.keras.applications import EfficientNetB0

# Layers and model building
from tensorflow.keras import layers, models

In [ ]:
# Dataset path
dataset_path = "../dataset"

In [ ]:
# Define image size
IMG_SIZE = 224

# Batch size
BATCH_SIZE = 32

# Create generator
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training dataset
train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Validation dataset
val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
# Load EfficientNetB0
base_model = EfficientNetB0(

    weights='imagenet',

    include_top=False,

    input_shape=(224,224,3)
)

In [ ]:
# Freeze all layers initially
base_model.trainable = False

In [ ]:
# Unfreeze last 20 layers
for layer in base_model.layers[-20:]:

    layer.trainable = True

In [ ]:
# Build classification model
model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(5, activation='softmax')
])

In [ ]:
# Compile model
model.compile(

    # Lower learning rate for fine-tuning
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

In [ ]:
# Train fine-tuned model
history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=10
)

In [ ]:
# Save fine-tuned model
model.save(
    "../models/efficientnet_finetuned.h5"
)